In [ ]:
import pandas as pd
import numpy as np
import string
from itertools import cycle

# Download raw data

In [ ]:
csv_file_path = '/path/to/files/data_raw_notsorted.csv'
df = pd.read_csv(csv_file_path)

excel_file_path = '/path/to/files/experiments_target_var_mapping.xlsx'
mapping_df = pd.read_excel(excel_file_path)

excel_file_path = '/path/to/files/tests_documentation.xlsx'
documentation_df = pd.read_excel(excel_file_path)

In [54]:
df_filtered = df.dropna()
df_filtered = df_filtered[df_filtered["experiment_id"].str.match(r"^\d{4}_\d{2}_\d{2}")]

In [55]:
unique_event_types = df_filtered['event_type'].unique()

for event in unique_event_types:
    df_filtered[event] = 0

for index, row in df_filtered.iterrows():
    event = row['event_type']
    df_filtered.at[index, event] = row['total_count']

In [56]:
event_columns = ['first_screen_view', 'email_enter_success','payment_screen_view', 'sale_confirmation_success']


In [57]:
grouped_df = df_filtered.groupby(['experiment_id', 'yyyymmdd','funnelId', "siteDomain"], as_index=False, sort=False).sum()

In [59]:
pd.Series(grouped_df['experiment_id'].unique()).to_excel('intermediate_data.xlsx')

In [ ]:
grouped_df['yyyymmdd'] = pd.to_datetime(grouped_df['yyyymmdd'], format='%Y%m%d')
mask = (grouped_df[event_columns] < 10).any(axis=1).to_numpy()
grouped_df = grouped_df.loc[~mask]
grouped_df['test_duration'] = grouped_df.groupby('experiment_id',sort=False)['yyyymmdd'].transform(lambda x: x.max() - x.min())


In [61]:
grouped_df = grouped_df[(grouped_df[['email_enter_success', 'payment_screen_view', 'sale_confirmation_success']].le(grouped_df['first_screen_view'], axis=0)).all(axis=1)]

# Prepare "distribution" dataframe

In [62]:
distribution_df = grouped_df

In [63]:
grouped = grouped_df.groupby(
    ['experiment_id', 'funnelId'],
    sort=False
)[
    ['first_screen_view', 'email_enter_success', 'payment_screen_view',
     'sale_confirmation_success', 'total_count']
].sum().reset_index()


# Prepare "sequential" dataframe

In [64]:
power_research_df = grouped_df.groupby(
    ['experiment_id','yyyymmdd','funnelId'],
    sort=False
)[
    ['first_screen_view', 'email_enter_success', 'payment_screen_view',
     'sale_confirmation_success', 'total_count']
].sum().reset_index()

power_research_df['Nth_Day'] = power_research_df.groupby(['experiment_id', 'funnelId'])['yyyymmdd'].rank(method='dense').astype(int)



In [ ]:
def freeze_and_mask(row):
    """add cumulative conversions for each day of the variable"""
    cum = 0
    freeze = False
    result = []
    for val in row:
        if not freeze:
            new_cum = cum + val
            if new_cum > cum:
                cum = new_cum
                result.append(cum)
            else:
                freeze = True
                result.append(np.nan)
        else:
            result.append(np.nan)
    return pd.Series(result, index=row.index)


def process_metric(df, metric, label):
    """create columns in dataframe"""
    pivot = df.pivot(index=['experiment_id', 'funnelId'], columns='Nth_Day', values=metric).fillna(0)
    pivot = pivot.sort_index(axis=1)
    frozen = pivot.apply(freeze_and_mask, axis=1)
    frozen.columns = [f'Cumulative_{label}_Day_{col}' for col in frozen.columns]
    return frozen

frozen_fsv = process_metric(power_research_df, 'first_screen_view', 'first_screen_view')
frozen_esv = process_metric(power_research_df, 'email_enter_success', 'email_enter_success')
frozen_psv = process_metric(power_research_df, 'payment_screen_view', 'payment_screen_view')
frozen_scs = process_metric(power_research_df, 'sale_confirmation_success', 'sale_confirmation_success')
frozen_all = pd.concat([frozen_fsv, frozen_esv, frozen_psv, frozen_scs], axis=1).reset_index()



# Add documentation to the existing dataframes

In [ ]:
final_grouped_df = pd.merge(grouped, grouped_df[["experiment_id","test_duration"]], on='experiment_id', how='left')


In [67]:
final_grouped_df.drop_duplicates(inplace=True)
frozen_all.drop_duplicates(inplace=True)

In [68]:
frozen_all['experiment_id'] = frozen_all['experiment_id'].astype(str).str.strip().str.lower()
documentation_df['experiment_id'] = documentation_df['experiment_id'].astype(str).str.strip().str.lower()


In [ ]:
frozen_all = frozen_all.reset_index(drop=True)

In [ ]:
final_grouped_df = pd.merge(final_grouped_df, documentation_df[["experiment_id","Result","funnelID", "Test Owner","Funnel"]], on='experiment_id', how='left')
sequential_final_df = pd.merge(frozen_all, documentation_df[["experiment_id","Result","funnelID", "Test Owner","Funnel"]], on='experiment_id', how='left')

In [ ]:
def extract_variation(row):
    """ extract details from exact test variation"""
    target_id = row['funnelId']
    mapping_str = row['funnelID']

    if pd.isna(mapping_str) or pd.isna(target_id):
        return None
    for line in mapping_str.splitlines():
        line = line.strip()
        if not line or ':' not in line:
            continue
        label, id_val = line.split(':', 1)
        if id_val.strip() == target_id:
            return label.strip()

    return None

final_grouped_df['variation'] = final_grouped_df.apply(extract_variation, axis=1)
sequential_final_df['variation'] = sequential_final_df.apply(extract_variation, axis=1)


In [ ]:
final_grouped_df = pd.merge(final_grouped_df, documentation_df[["experiment_id","Result","funnelID"]], on='experiment_id', how='left')

In [73]:
final_grouped_df = pd.merge(final_grouped_df, mapping_df[['experiment_id',"target_var"]], on='experiment_id', how='left')
sequential_final_df = pd.merge(sequential_final_df, mapping_df[['experiment_id',"target_var"]], on='experiment_id', how='left')

In [ ]:
final_grouped_df = final_grouped_df[final_grouped_df["target_var"] != "bad"]
sequential_final_df = sequential_final_df[sequential_final_df["target_var"] != "bad"]

In [75]:
final_grouped_df = final_grouped_df[['experiment_id','funnelId','Result_x',"Test Owner", "Funnel", 'variation','target_var','first_screen_view','email_enter_success','payment_screen_view','sale_confirmation_success','test_duration']].dropna()

# Add Conersion rates to right variables

targeting variables for the test


In [76]:
mapping_dict = {
    "fpv": "first_screen_view",
    "psv": "payment_screen_view",
    "scs": "sale_confirmation_success",
    "fsv": "first_screen_view"
}

In [78]:
indexed_sequential_df = sequential_final_df.set_index(['experiment_id', 'funnelId'])

def mask_row(row, keep_keywords):
    for col in row.index:
        if "Cumulative_" in col and not any(k in col for k in keep_keywords):
            row[col] = np.nan
    return row

all_keep_keywords = ['first_screen_view', 'payment_screen_view', 'sale_confirmation_success']
masked_sequential_df = sequential_final_df.copy()
masked_sequential_df = masked_sequential_df.apply(lambda row: mask_row(row, all_keep_keywords), axis=1)
indexed_masked_df = masked_sequential_df.set_index(['experiment_id', 'funnelId'])

def assign_conversion_rates(row, var1="first_screen_view", var2="payment_screen_view", trigger=0):
    if trigger == 0:
        mapping_str = row.get("target_var")
        if not mapping_str or not isinstance(mapping_str, str):
            return row

        tokens = mapping_str.strip().split()
        if len(tokens) == 2:
            token1, token2 = tokens[0].lower(), tokens[1].lower()
            col1 = mapping_dict.get(token1, var1)
            col2 = mapping_dict.get(token2, var2)
        else:
            col1 = var1
            col2 = var2
    else:
        col1 = var1
        col2 = var2

    row_key = (row['experiment_id'], row['funnelId'])

    if row_key in indexed_masked_df.index:
        row["start"] = col1
        row["finish"] = col2
        row["CR_1"] = row.get(col1)
        row["CR_2"] = row.get(col2)
        row["conversion_rate"] = (
            row["CR_2"] / row["CR_1"]
            if row["CR_1"] and pd.notna(row["CR_1"]) else None
        )
    return row


In [79]:

final_grouped_df = final_grouped_df.apply(assign_conversion_rates, axis=1)
final_grouped_df_SCS = final_grouped_df.apply(
    lambda row: assign_conversion_rates(row, var1="first_screen_view", var2="sale_confirmation_success", trigger=1), axis=1
)

In [ ]:
final_grouped_df = final_grouped_df.dropna(axis=1, how='all')
sequential_final_df = sequential_final_df.dropna(axis=1, how='all')
final_grouped_df_SCS = final_grouped_df_SCS.dropna(axis=1, how='all')

In [ ]:
standardization_map = {
    "Funnel_B": ['here were funnel values'],

    'Funnel_A':['here were funnel values'],

    'Funnel_T':['here were funnel values'],

    'Funnel_Q':['here were funnel values']
}
reverse_map = {}
for key, values in standardization_map.items():
    for v in values:
        reverse_map[v] = key

In [83]:
final_grouped_df["funnel"] = final_grouped_df["Funnel"].map(reverse_map)
sequential_final_df["funnel"] = sequential_final_df["Funnel"].map(reverse_map)
final_grouped_df_SCS["funnel"] = final_grouped_df_SCS["Funnel"].map(reverse_map)
grouped_df["funnel"] = grouped_df["siteDomain"].map(reverse_map)
grouped_df["sitedomain"] = grouped_df["siteDomain"].map(reverse_map)
distribution_df["sitedomain"] = distribution_df["siteDomain"].map(reverse_map)

In [84]:
final_grouped_df[final_grouped_df.funnel=="Funnel_A"]["experiment_id"].unique().shape[0]

21

In [ ]:

def generate_labels(exp_id, prefix, i):
  return [f"{exp_id[:10]}_{prefix}_{i}"]


unique_experiments = pd.concat([
    final_grouped_df['experiment_id'],
    sequential_final_df['experiment_id'],
    grouped_df['experiment_id'],
    distribution_df['experiment_id']
]).unique()
exp_labels = []
for i, exp_id in enumerate(unique_experiments):
    label = generate_labels(exp_id, "experiment", i)[0]
    exp_labels.append(label)

exp_map = dict(zip(unique_experiments, exp_labels))

final_grouped_df['experiment_id_anon'] = final_grouped_df['experiment_id'].map(exp_map)
sequential_final_df['experiment_id_anon'] = sequential_final_df['experiment_id'].map(exp_map)
final_grouped_df_SCS['experiment_id_anon'] = final_grouped_df_SCS['experiment_id'].map(exp_map)
grouped_df["experiment_id_anon"] = grouped_df["experiment_id"].map(exp_map)
distribution_df["experiment_id_anon"] = distribution_df["experiment_id"].map(exp_map)


In [ ]:
exp_map2 = {
    key: value
    for key, value in exp_map.items()
    if key in set(final_grouped_df["experiment_id"])
}

In [87]:
sequential_anonimized_df= sequential_final_df.drop(columns=['experiment_id', 'Funnel'])
sequential_anonimized_df = sequential_anonimized_df.rename(columns={'experiment_id_anon': 'experiment_id'})

In [88]:
final_anonimized_df= final_grouped_df.drop(columns=['experiment_id', 'Funnel'])
final_anonimized_df = final_anonimized_df.rename(columns={'experiment_id_anon': 'experiment_id'})

In [89]:
final_anonimized_df_SCS= final_grouped_df_SCS.drop(columns=['experiment_id', 'Funnel'])
final_anonimized_df_SCS = final_anonimized_df_SCS.rename(columns={'experiment_id_anon': 'experiment_id'})

In [90]:
grouped_df = grouped_df.drop(columns=['experiment_id', 'siteDomain'])
distribution_df = distribution_df.drop(columns=['experiment_id', 'siteDomain'])

grouped_df = grouped_df.rename(columns={'experiment_id_anon': 'experiment_id'})
grouped_df = grouped_df.rename(columns={'sitedomain': 'siteDomain'})

distribution_df = distribution_df.rename(columns={'experiment_id_anon': 'experiment_id'})
distribution_df = distribution_df.rename(columns={'sitedomain': 'siteDomain'})

In [ ]:

final_anonimized_df.to_excel("/path/to/files/final_anonimous_dataset.xlsx")
sequential_anonimized_df.to_excel("/path/to/files/final_anonimous_sequential_dataset.xlsx")
final_anonimized_df_SCS.to_excel("/path/to/files/final_anonimous_SCS_dataset.xlsx")
grouped_df.to_excel("/path/to/files/grouped_df.xlsx")
distribution_df.to_excel("/path/to/files/distribution_df.xlsx")